In [1]:
import torch
import torch.nn as nn
from torch import Tensor
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader,SubsetRandomSampler,ConcatDataset
import torch.nn.functional as F

from model_v2_compatible import SeqNN

In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [3]:
model = SeqNN()
model = model.to(device)

In [4]:
model.eval()

SeqNN(
  (stochastic_reverse_complement): StochasticReverseComplement()
  (stochastic_shift): StochasticShift()
  (conv_block_1): ConvBlock(
    (conv): Conv1d(4, 128, kernel_size=(15,), stride=(1,), padding=(7,), bias=False)
    (batch_norm): BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_tower): ConvTower(
    (conv_tower): Sequential(
      (0): ReLU()
      (1): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ReLU()
      (5): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (7): MaxPool1d(kernel_size=2, stride=2, paddi

In [5]:
from torchinfo import summary

summary(model, input_size=(2, 4, 1310720), col_names=["output_size", "num_params"])

Layer (type:depth-idx)                   Output Shape              Param #
SeqNN                                    [2, 130305, 1]            --
├─StochasticReverseComplement: 1-1       [2, 4, 1310720]           --
├─StochasticShift: 1-2                   [2, 4, 1310720]           --
├─ConvBlock: 1-3                         [2, 128, 655360]          --
│    └─Conv1d: 2-1                       [2, 128, 1310720]         7,680
│    └─BatchNorm1d: 2-2                  [2, 128, 1310720]         256
│    └─MaxPool1d: 2-3                    [2, 128, 655360]          --
├─ConvTower: 1-4                         [2, 128, 640]             --
│    └─Sequential: 2-4                   [2, 128, 640]             --
│    │    └─ReLU: 3-1                    [2, 128, 655360]          --
│    │    └─Conv1d: 3-2                  [2, 128, 655360]          81,920
│    │    └─BatchNorm1d: 3-3             [2, 128, 655360]          256
│    │    └─MaxPool1d: 3-4               [2, 128, 327680]          --
│    │

In [6]:
for name, module in model.named_modules():
    print(name, "->", module)

 -> SeqNN(
  (stochastic_reverse_complement): StochasticReverseComplement()
  (stochastic_shift): StochasticShift()
  (conv_block_1): ConvBlock(
    (conv): Conv1d(4, 128, kernel_size=(15,), stride=(1,), padding=(7,), bias=False)
    (batch_norm): BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_tower): ConvTower(
    (conv_tower): Sequential(
      (0): ReLU()
      (1): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ReLU()
      (5): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (7): MaxPool1d(kernel_size=2, stride=2, p

In [7]:
state_dict = model.state_dict()

In [8]:
for key in state_dict.keys():
    print(key)

conv_block_1.conv.weight
conv_block_1.batch_norm.weight
conv_block_1.batch_norm.bias
conv_block_1.batch_norm.running_mean
conv_block_1.batch_norm.running_var
conv_block_1.batch_norm.num_batches_tracked
conv_tower.conv_tower.1.weight
conv_tower.conv_tower.2.weight
conv_tower.conv_tower.2.bias
conv_tower.conv_tower.2.running_mean
conv_tower.conv_tower.2.running_var
conv_tower.conv_tower.2.num_batches_tracked
conv_tower.conv_tower.5.weight
conv_tower.conv_tower.6.weight
conv_tower.conv_tower.6.bias
conv_tower.conv_tower.6.running_mean
conv_tower.conv_tower.6.running_var
conv_tower.conv_tower.6.num_batches_tracked
conv_tower.conv_tower.9.weight
conv_tower.conv_tower.10.weight
conv_tower.conv_tower.10.bias
conv_tower.conv_tower.10.running_mean
conv_tower.conv_tower.10.running_var
conv_tower.conv_tower.10.num_batches_tracked
conv_tower.conv_tower.13.weight
conv_tower.conv_tower.14.weight
conv_tower.conv_tower.14.bias
conv_tower.conv_tower.14.running_mean
conv_tower.conv_tower.14.running_var


In [9]:
import h5py

In [10]:
# h5_file = h5py.File("/project/fudenber_735/tensorflow_models/akita/v1/model_best.h5", "r") # akita.V1
h5_file = h5py.File("/project/fudenber_735/tensorflow_models/akita/v2/models/f0c0/train/model1_best.h5", "r") #akita.V2, model 0, mouse

In [11]:
print(list(h5_file["model_weights"].keys()))

['add', 'add_1', 'add_10', 'add_11', 'add_12', 'add_13', 'add_14', 'add_15', 'add_16', 'add_2', 'add_3', 'add_4', 'add_5', 'add_6', 'add_7', 'add_8', 'add_9', 'batch_normalization', 'batch_normalization_1', 'batch_normalization_10', 'batch_normalization_11', 'batch_normalization_12', 'batch_normalization_13', 'batch_normalization_14', 'batch_normalization_15', 'batch_normalization_16', 'batch_normalization_17', 'batch_normalization_18', 'batch_normalization_19', 'batch_normalization_2', 'batch_normalization_20', 'batch_normalization_21', 'batch_normalization_22', 'batch_normalization_23', 'batch_normalization_24', 'batch_normalization_25', 'batch_normalization_26', 'batch_normalization_27', 'batch_normalization_28', 'batch_normalization_29', 'batch_normalization_3', 'batch_normalization_30', 'batch_normalization_31', 'batch_normalization_32', 'batch_normalization_33', 'batch_normalization_34', 'batch_normalization_35', 'batch_normalization_36', 'batch_normalization_37', 'batch_normaliz

In [12]:
def assign_conv_weights(h5_file, tf_layer_path, pytorch_conv_layer):
    """
    Assign weights from a TensorFlow convolutional layer to a PyTorch convolutional layer.
    
    Parameters:
        h5_file (h5py.File): The HDF5 file containing TensorFlow weights.
        tf_layer_path (str): Path to the TensorFlow layer in the HDF5 file.
        pytorch_conv_layer (torch.nn.Conv1d or torch.nn.Conv2d): The PyTorch convolutional layer.

    """
    # Access TensorFlow kernel weights
    tf_weights = h5_file[tf_layer_path]['kernel:0'][:]
    
    # Convert to PyTorch tensor
    pytorch_weights = torch.tensor(tf_weights, dtype=torch.float32)
    
    # Transpose TensorFlow weights to match PyTorch's format
    pytorch_weights = pytorch_weights.permute(2, 1, 0)
    
    # Ensure the shapes match
    assert pytorch_weights.shape == pytorch_conv_layer.weight.data.shape, \
        f"Shape mismatch: {pytorch_weights.shape} vs {pytorch_conv_layer.weight.data.shape}"
    
    # Assign weights to the PyTorch layer
    pytorch_conv_layer.weight.data = pytorch_weights

    print(f"Assigned weights to PyTorch layer: {pytorch_conv_layer}")

In [13]:
def assign_batch_norm_weights(h5_file, tf_layer_path, pytorch_batch_norm_layer):
    """
    Assign batch normalization weights from TensorFlow to PyTorch.
    
    Parameters:
        h5_file (h5py.File): The HDF5 file containing TensorFlow weights.
        tf_layer_path (str): Path to the TensorFlow batch normalization layer in the HDF5 file.
        pytorch_batch_norm_layer (torch.nn.BatchNorm1d or torch.nn.BatchNorm2d): The PyTorch batch normalization layer.
    """
    # Access TensorFlow BatchNorm parameters
    batch_norm_group = h5_file[tf_layer_path]

    # Extract TensorFlow parameters
    beta = batch_norm_group["beta:0"][:]  # Bias (beta in TensorFlow)
    gamma = batch_norm_group["gamma:0"][:]  # Scale (gamma in TensorFlow)
    moving_mean = batch_norm_group["moving_mean:0"][:]  # Running mean
    moving_variance = batch_norm_group["moving_variance:0"][:]  # Running variance

    # Convert to PyTorch tensors
    beta_tensor = torch.tensor(beta, dtype=torch.float32)
    gamma_tensor = torch.tensor(gamma, dtype=torch.float32)
    moving_mean_tensor = torch.tensor(moving_mean, dtype=torch.float32)
    moving_variance_tensor = torch.tensor(moving_variance, dtype=torch.float32)

    # Assign values to the PyTorch BatchNorm layer
    pytorch_batch_norm_layer.bias.data = beta_tensor
    pytorch_batch_norm_layer.weight.data = gamma_tensor
    pytorch_batch_norm_layer.running_mean.data = moving_mean_tensor
    pytorch_batch_norm_layer.running_var.data = moving_variance_tensor

    print(f"Assigned batch normalization weights to PyTorch layer: {pytorch_batch_norm_layer}")


In [14]:
# ConvBlock

assign_conv_weights(h5_file, "model_weights/conv1d/conv1d", model.conv_block_1.conv)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization/batch_normalization", model.conv_block_1.batch_norm)

Assigned weights to PyTorch layer: Conv1d(4, 128, kernel_size=(15,), stride=(1,), padding=(7,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)


In [15]:
# ConvTower
# 1
assign_conv_weights(h5_file, "model_weights/conv1d_1/conv1d_1", model.conv_tower.conv_tower[1])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_1/batch_normalization_1", model.conv_tower.conv_tower[2])

Assigned weights to PyTorch layer: Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)


In [16]:
# 2
assign_conv_weights(h5_file, "model_weights/conv1d_2/conv1d_2", model.conv_tower.conv_tower[5])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_2/batch_normalization_2", model.conv_tower.conv_tower[6])

Assigned weights to PyTorch layer: Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)


In [17]:
# 3
assign_conv_weights(h5_file, "model_weights/conv1d_3/conv1d_3", model.conv_tower.conv_tower[9])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_3/batch_normalization_3", model.conv_tower.conv_tower[10])

Assigned weights to PyTorch layer: Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)


In [18]:
# 4
assign_conv_weights(h5_file, "model_weights/conv1d_4/conv1d_4", model.conv_tower.conv_tower[13])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_4/batch_normalization_4", model.conv_tower.conv_tower[14])

Assigned weights to PyTorch layer: Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)


In [19]:
# 5
assign_conv_weights(h5_file, "model_weights/conv1d_5/conv1d_5", model.conv_tower.conv_tower[17])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_5/batch_normalization_5", model.conv_tower.conv_tower[18])

Assigned weights to PyTorch layer: Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)


In [20]:
# 6
assign_conv_weights(h5_file, "model_weights/conv1d_6/conv1d_6", model.conv_tower.conv_tower[21])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_6/batch_normalization_6", model.conv_tower.conv_tower[22])

Assigned weights to PyTorch layer: Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)


In [21]:
# 7
assign_conv_weights(h5_file, "model_weights/conv1d_7/conv1d_7", model.conv_tower.conv_tower[25])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_7/batch_normalization_7", model.conv_tower.conv_tower[26])

Assigned weights to PyTorch layer: Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)


In [22]:
# 8
assign_conv_weights(h5_file, "model_weights/conv1d_8/conv1d_8", model.conv_tower.conv_tower[29])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_8/batch_normalization_8", model.conv_tower.conv_tower[30])

Assigned weights to PyTorch layer: Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)


In [23]:
# 9
assign_conv_weights(h5_file, "model_weights/conv1d_9/conv1d_9", model.conv_tower.conv_tower[33])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_9/batch_normalization_9", model.conv_tower.conv_tower[34])

Assigned weights to PyTorch layer: Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)


In [24]:
# 10
assign_conv_weights(h5_file, "model_weights/conv1d_10/conv1d_10", model.conv_tower.conv_tower[37])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_10/batch_normalization_10", model.conv_tower.conv_tower[38])

Assigned weights to PyTorch layer: Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)


In [25]:
# ResidualDilatedBlock

# 1
assign_conv_weights(h5_file, "model_weights/conv1d_11/conv1d_11", model.residual1d_block1.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_11/batch_normalization_11", model.residual1d_block1.norm1)

assign_conv_weights(h5_file, "model_weights/conv1d_12/conv1d_12", model.residual1d_block1.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_12/batch_normalization_12", model.residual1d_block1.norm2)

Assigned weights to PyTorch layer: Conv1d(128, 64, kernel_size=(3,), stride=(1,), padding=(1,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
Assigned weights to PyTorch layer: Conv1d(64, 128, kernel_size=(1,), stride=(1,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)


In [26]:
# 2
assign_conv_weights(h5_file, "model_weights/conv1d_13/conv1d_13", model.residual1d_block2.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_13/batch_normalization_13", model.residual1d_block2.norm1)

assign_conv_weights(h5_file, "model_weights/conv1d_14/conv1d_14", model.residual1d_block2.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_14/batch_normalization_14", model.residual1d_block2.norm2)

Assigned weights to PyTorch layer: Conv1d(128, 64, kernel_size=(3,), stride=(1,), padding=(2,), dilation=(2,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
Assigned weights to PyTorch layer: Conv1d(64, 128, kernel_size=(1,), stride=(1,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)


In [27]:
# 3
assign_conv_weights(h5_file, "model_weights/conv1d_15/conv1d_15", model.residual1d_block3.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_15/batch_normalization_15", model.residual1d_block3.norm1)

assign_conv_weights(h5_file, "model_weights/conv1d_16/conv1d_16", model.residual1d_block3.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_16/batch_normalization_16", model.residual1d_block3.norm2)

Assigned weights to PyTorch layer: Conv1d(128, 64, kernel_size=(3,), stride=(1,), padding=(3,), dilation=(3,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
Assigned weights to PyTorch layer: Conv1d(64, 128, kernel_size=(1,), stride=(1,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)


In [28]:
# 4
assign_conv_weights(h5_file, "model_weights/conv1d_17/conv1d_17", model.residual1d_block4.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_17/batch_normalization_17", model.residual1d_block4.norm1)

assign_conv_weights(h5_file, "model_weights/conv1d_18/conv1d_18", model.residual1d_block4.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_18/batch_normalization_18", model.residual1d_block4.norm2)

Assigned weights to PyTorch layer: Conv1d(128, 64, kernel_size=(3,), stride=(1,), padding=(5,), dilation=(5,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
Assigned weights to PyTorch layer: Conv1d(64, 128, kernel_size=(1,), stride=(1,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)


In [29]:
# 5
assign_conv_weights(h5_file, "model_weights/conv1d_19/conv1d_19", model.residual1d_block5.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_19/batch_normalization_19", model.residual1d_block5.norm1)

assign_conv_weights(h5_file, "model_weights/conv1d_20/conv1d_20", model.residual1d_block5.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_20/batch_normalization_20", model.residual1d_block5.norm2)

Assigned weights to PyTorch layer: Conv1d(128, 64, kernel_size=(3,), stride=(1,), padding=(8,), dilation=(8,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
Assigned weights to PyTorch layer: Conv1d(64, 128, kernel_size=(1,), stride=(1,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)


In [30]:
# 6
assign_conv_weights(h5_file, "model_weights/conv1d_21/conv1d_21", model.residual1d_block6.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_21/batch_normalization_21", model.residual1d_block6.norm1)

assign_conv_weights(h5_file, "model_weights/conv1d_22/conv1d_22", model.residual1d_block6.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_22/batch_normalization_22", model.residual1d_block6.norm2)

Assigned weights to PyTorch layer: Conv1d(128, 64, kernel_size=(3,), stride=(1,), padding=(13,), dilation=(13,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
Assigned weights to PyTorch layer: Conv1d(64, 128, kernel_size=(1,), stride=(1,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)


In [31]:
# 7
assign_conv_weights(h5_file, "model_weights/conv1d_23/conv1d_23", model.residual1d_block7.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_23/batch_normalization_23", model.residual1d_block7.norm1)

assign_conv_weights(h5_file, "model_weights/conv1d_24/conv1d_24", model.residual1d_block7.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_24/batch_normalization_24", model.residual1d_block7.norm2)

Assigned weights to PyTorch layer: Conv1d(128, 64, kernel_size=(3,), stride=(1,), padding=(21,), dilation=(21,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
Assigned weights to PyTorch layer: Conv1d(64, 128, kernel_size=(1,), stride=(1,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)


In [32]:
# 8
assign_conv_weights(h5_file, "model_weights/conv1d_25/conv1d_25", model.residual1d_block8.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_25/batch_normalization_25", model.residual1d_block8.norm1)

assign_conv_weights(h5_file, "model_weights/conv1d_26/conv1d_26", model.residual1d_block8.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_26/batch_normalization_26", model.residual1d_block8.norm2)

Assigned weights to PyTorch layer: Conv1d(128, 64, kernel_size=(3,), stride=(1,), padding=(34,), dilation=(34,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
Assigned weights to PyTorch layer: Conv1d(64, 128, kernel_size=(1,), stride=(1,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)


In [33]:
# 9
assign_conv_weights(h5_file, "model_weights/conv1d_27/conv1d_27", model.residual1d_block9.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_27/batch_normalization_27", model.residual1d_block9.norm1)

assign_conv_weights(h5_file, "model_weights/conv1d_28/conv1d_28", model.residual1d_block9.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_28/batch_normalization_28", model.residual1d_block9.norm2)

Assigned weights to PyTorch layer: Conv1d(128, 64, kernel_size=(3,), stride=(1,), padding=(55,), dilation=(55,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
Assigned weights to PyTorch layer: Conv1d(64, 128, kernel_size=(1,), stride=(1,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)


In [34]:
# 10
assign_conv_weights(h5_file, "model_weights/conv1d_29/conv1d_29", model.residual1d_block10.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_29/batch_normalization_29", model.residual1d_block10.norm1)

assign_conv_weights(h5_file, "model_weights/conv1d_30/conv1d_30", model.residual1d_block10.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_30/batch_normalization_30", model.residual1d_block10.norm2)

Assigned weights to PyTorch layer: Conv1d(128, 64, kernel_size=(3,), stride=(1,), padding=(89,), dilation=(89,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
Assigned weights to PyTorch layer: Conv1d(64, 128, kernel_size=(1,), stride=(1,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)


In [35]:
# 11
assign_conv_weights(h5_file, "model_weights/conv1d_31/conv1d_31", model.residual1d_block11.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_31/batch_normalization_31", model.residual1d_block11.norm1)

assign_conv_weights(h5_file, "model_weights/conv1d_32/conv1d_32", model.residual1d_block11.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_32/batch_normalization_32", model.residual1d_block11.norm2)

Assigned weights to PyTorch layer: Conv1d(128, 64, kernel_size=(3,), stride=(1,), padding=(145,), dilation=(145,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
Assigned weights to PyTorch layer: Conv1d(64, 128, kernel_size=(1,), stride=(1,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)


In [36]:
# ConvBlockReduce
assign_conv_weights(h5_file, "model_weights/conv1d_33/conv1d_33", model.conv_reduce.layers[1])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_33/batch_normalization_33", model.conv_reduce.layers[2])

Assigned weights to PyTorch layer: Conv1d(128, 80, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm1d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)


In [37]:
def assign_conv2d_weights(h5_file, tf_layer_path, pytorch_conv2d_layer):
    """
    Assign weights from a TensorFlow Conv2d layer to a PyTorch Conv2d layer.
    
    Parameters:
        h5_file (h5py.File): The HDF5 file containing TensorFlow weights.
        tf_layer_path (str): Path to the TensorFlow Conv2d layer in the HDF5 file.
        pytorch_conv2d_layer (torch.nn.Conv2d): The PyTorch Conv2d layer.
    """
    # Access TensorFlow Conv2D weights
    tf_weights = h5_file[tf_layer_path]['kernel:0'][:]
    
    # Convert to PyTorch tensor
    pytorch_weights = torch.tensor(tf_weights, dtype=torch.float32)
    
    # Transpose TensorFlow weights to match PyTorch's format
    pytorch_weights = pytorch_weights.permute(3, 2, 0, 1)  # [output_channels, input_channels, filter_height, filter_width]
    
    # Ensure the shapes match
    assert pytorch_weights.shape == pytorch_conv2d_layer.weight.data.shape, \
        f"Shape mismatch: {pytorch_weights.shape} vs {pytorch_conv2d_layer.weight.data.shape}"
    
    # Assign weights to the PyTorch layer
    pytorch_conv2d_layer.weight.data = pytorch_weights

    print(f"Assigned Conv2D weights to PyTorch layer: {pytorch_conv2d_layer}")

In [38]:
# HEAD

assign_conv2d_weights(h5_file, "model_weights/conv2d/conv2d", model.conv2d_block.block[1])
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_34/batch_normalization_34", model.conv2d_block.block[2])

Assigned Conv2D weights to PyTorch layer: Conv2d(80, 80, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)


In [39]:
# ResidualDilatedBlock - 2D

# 1
assign_conv2d_weights(h5_file, "model_weights/conv2d_1/conv2d_1", model.residual2d_block1.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_35/batch_normalization_35", model.residual2d_block1.norm1)

assign_conv2d_weights(h5_file, "model_weights/conv2d_2/conv2d_2", model.residual2d_block1.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_36/batch_normalization_36", model.residual2d_block1.norm2)

Assigned Conv2D weights to PyTorch layer: Conv2d(80, 40, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm2d(40, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
Assigned Conv2D weights to PyTorch layer: Conv2d(40, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)


In [40]:
# 2
assign_conv2d_weights(h5_file, "model_weights/conv2d_3/conv2d_3", model.residual2d_block2.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_37/batch_normalization_37", model.residual2d_block2.norm1)

assign_conv2d_weights(h5_file, "model_weights/conv2d_4/conv2d_4", model.residual2d_block2.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_38/batch_normalization_38", model.residual2d_block2.norm2)

Assigned Conv2D weights to PyTorch layer: Conv2d(80, 40, kernel_size=(3, 3), stride=(1, 1), padding=(2, 2), dilation=(2, 2), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm2d(40, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
Assigned Conv2D weights to PyTorch layer: Conv2d(40, 80, kernel_size=(1, 1), stride=(1, 1), dilation=(2, 2), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)


In [41]:
# 3
assign_conv2d_weights(h5_file, "model_weights/conv2d_5/conv2d_5", model.residual2d_block3.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_39/batch_normalization_39", model.residual2d_block3.norm1)

assign_conv2d_weights(h5_file, "model_weights/conv2d_6/conv2d_6", model.residual2d_block3.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_40/batch_normalization_40", model.residual2d_block3.norm2)

Assigned Conv2D weights to PyTorch layer: Conv2d(80, 40, kernel_size=(3, 3), stride=(1, 1), padding=(4, 4), dilation=(4, 4), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm2d(40, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
Assigned Conv2D weights to PyTorch layer: Conv2d(40, 80, kernel_size=(1, 1), stride=(1, 1), dilation=(4, 4), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)


In [42]:
# 4
assign_conv2d_weights(h5_file, "model_weights/conv2d_7/conv2d_7", model.residual2d_block4.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_41/batch_normalization_41", model.residual2d_block4.norm1)

assign_conv2d_weights(h5_file, "model_weights/conv2d_8/conv2d_8", model.residual2d_block4.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_42/batch_normalization_42", model.residual2d_block4.norm2)

Assigned Conv2D weights to PyTorch layer: Conv2d(80, 40, kernel_size=(3, 3), stride=(1, 1), padding=(7, 7), dilation=(7, 7), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm2d(40, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
Assigned Conv2D weights to PyTorch layer: Conv2d(40, 80, kernel_size=(1, 1), stride=(1, 1), dilation=(7, 7), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)


In [43]:
# 5
assign_conv2d_weights(h5_file, "model_weights/conv2d_9/conv2d_9", model.residual2d_block5.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_43/batch_normalization_43", model.residual2d_block5.norm1)

assign_conv2d_weights(h5_file, "model_weights/conv2d_10/conv2d_10", model.residual2d_block5.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_44/batch_normalization_44", model.residual2d_block5.norm2)

Assigned Conv2D weights to PyTorch layer: Conv2d(80, 40, kernel_size=(3, 3), stride=(1, 1), padding=(12, 12), dilation=(12, 12), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm2d(40, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
Assigned Conv2D weights to PyTorch layer: Conv2d(40, 80, kernel_size=(1, 1), stride=(1, 1), dilation=(12, 12), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)


In [44]:
# 6
assign_conv2d_weights(h5_file, "model_weights/conv2d_11/conv2d_11", model.residual2d_block6.conv1)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_45/batch_normalization_45", model.residual2d_block6.norm1)

assign_conv2d_weights(h5_file, "model_weights/conv2d_12/conv2d_12", model.residual2d_block6.conv2)
assign_batch_norm_weights(h5_file, "model_weights/batch_normalization_46/batch_normalization_46", model.residual2d_block6.norm2)

Assigned Conv2D weights to PyTorch layer: Conv2d(80, 40, kernel_size=(3, 3), stride=(1, 1), padding=(21, 21), dilation=(21, 21), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm2d(40, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
Assigned Conv2D weights to PyTorch layer: Conv2d(40, 80, kernel_size=(1, 1), stride=(1, 1), dilation=(21, 21), bias=False)
Assigned batch normalization weights to PyTorch layer: BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)


In [45]:
def assign_dense_weights(h5_file, tf_layer_path, pytorch_dense_layer):
    """
    Assign weights and biases from a TensorFlow Dense layer to a PyTorch Dense layer.
    
    Parameters:
        h5_file (h5py.File): The HDF5 file containing TensorFlow weights.
        tf_layer_path (str): Path to the TensorFlow dense layer in the HDF5 file.
        pytorch_dense_layer (torch.nn.Linear): The PyTorch dense (fully connected) layer.
    """
    # Access TensorFlow Dense layer parameters (weights and biases)
    tf_kernel = h5_file[tf_layer_path]['kernel:0'][:]  # Weights (input_units, output_units)
    tf_bias = h5_file[tf_layer_path]['bias:0'][:]      # Bias (output_units,)
    
    # Convert to PyTorch tensors
    pytorch_weights = torch.tensor(tf_kernel, dtype=torch.float32)
    pytorch_bias = torch.tensor(tf_bias, dtype=torch.float32)
    
    # Transpose TensorFlow weights to match PyTorch's format
    pytorch_weights = pytorch_weights.t()  # (output_units, input_units)
    
    # Ensure the shapes match
    assert pytorch_weights.shape == pytorch_dense_layer.weight.data.shape, \
        f"Shape mismatch: {pytorch_weights.shape} vs {pytorch_dense_layer.weight.data.shape}"
    assert pytorch_bias.shape == pytorch_dense_layer.bias.data.shape, \
        f"Shape mismatch: {pytorch_bias.shape} vs {pytorch_dense_layer.bias.data.shape}"
    
    # Assign weights and biases to the PyTorch layer
    pytorch_dense_layer.weight.data = pytorch_weights
    pytorch_dense_layer.bias.data = pytorch_bias

    print(f"Assigned Dense layer weights and biases to PyTorch layer: {pytorch_dense_layer}")


In [46]:
assign_batch_norm_weights(h5_file, "model_weights/squeeze_excite/squeeze_excite/batch_normalization", model.squeeze_excite.norm)

Assigned batch normalization weights to PyTorch layer: BatchNorm1d(80, eps=0.001, momentum=0.9, affine=True, track_running_stats=True)


In [47]:
assign_dense_weights(h5_file, "model_weights/squeeze_excite/squeeze_excite/dense", model.squeeze_excite.dense1)

Assigned Dense layer weights and biases to PyTorch layer: Linear(in_features=80, out_features=10, bias=True)


In [48]:
assign_dense_weights(h5_file, "model_weights/squeeze_excite/squeeze_excite/dense_1", model.squeeze_excite.dense2)

Assigned Dense layer weights and biases to PyTorch layer: Linear(in_features=10, out_features=80, bias=True)


In [49]:
pytorch_dense_layer = model.final.dense

# Access TensorFlow Dense layer parameters (weights and biases)
tf_kernel = h5_file["model_weights/dense_1/dense_1"]['kernel:0'][:,0]  # Weights (input_units, output_units)
tf_bias = h5_file["model_weights/dense_1/dense_1"]['bias:0'][0]      # Bias (output_units,)

# Convert to PyTorch tensors
pytorch_weights = torch.tensor(tf_kernel, dtype=torch.float32)
pytorch_bias = torch.tensor(tf_bias, dtype=torch.float32)

# Transpose TensorFlow weights to match PyTorch's format
pytorch_weights = pytorch_weights.t()  # (output_units, input_units)

# # Ensure the shapes match
# assert pytorch_weights.shape == pytorch_dense_layer.weight.data.shape, \
#     f"Shape mismatch: {pytorch_weights.shape} vs {pytorch_dense_layer.weight.data.shape}"
# assert pytorch_bias.shape == pytorch_dense_layer.bias.data.shape, \
#     f"Shape mismatch: {pytorch_bias.shape} vs {pytorch_dense_layer.bias.data.shape}"

# Assign weights and biases to the PyTorch layer
pytorch_dense_layer.weight.data = pytorch_weights
pytorch_dense_layer.bias.data = pytorch_bias

print(f"Assigned Dense layer weights and biases to PyTorch layer: {pytorch_dense_layer}")

Assigned Dense layer weights and biases to PyTorch layer: Linear(in_features=80, out_features=1, bias=True)


In [50]:
model.final.dense

Linear(in_features=80, out_features=1, bias=True)

In [ ]:
# Final
# assign_dense_weights(h5_file, "model_weights/dense/dense", model.final.dense)

In [51]:
# Save the entire model
torch.save(model, "model_v2_mouse_model0_target0.pth")